# Import

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Transfer dataset - Top 5 leagues

## Loading the DataFrame

In [5]:
df1 = pd.read_csv('../data/raw/transfer/01_premier_league.csv')
df2 = pd.read_csv('../data/raw/transfer/02_la_liga.csv')
df3 = pd.read_csv('../data/raw/transfer/03_bundesliga.csv')
df4 = pd.read_csv('../data/raw/transfer/04_ligue_1.csv')
df5 = pd.read_csv('../data/raw/transfer/05_serie_a.csv')

# Filtering dataset for scope

In [6]:
def clean_transfer_data(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    # keep only years in scope
    df = df[(df["year"] >= 2013) & (df["year"] <= 2022)]

    # keep only incoming transfers
    df = df[df["transfer_movement"] == "in"]

    # drop rows where transfer fee is missing
    df = df.dropna(subset=["fee_cleaned"])

    # drop unnecessary columns
    cols_to_drop = [
        "player_name",
        "age",
        "position",
        "club_involved_name",
        "fee",
        "transfer_period",
    ]
    df = df.drop(columns=cols_to_drop)

    return df


# paths
raw_path = Path("../data/raw/transfer")
intermediate_path = Path("../data/intermediate")
intermediate_path.mkdir(parents=True, exist_ok=True)

# files
league_files = [
    "01_premier_league.csv",
    "02_la_liga.csv",
    "03_bundesliga.csv",
    "04_ligue_1.csv",
    "05_serie_a.csv",
]

# clean and save as csv
cleaned_dfs = {}

for file_name in league_files:
    file_path = raw_path / file_name
    cleaned_df = clean_transfer_data(file_path)

    cleaned_dfs[file_name] = cleaned_df

    output_name = file_name.replace(".csv", "_filtered.csv")
    cleaned_df.to_csv(intermediate_path / output_name, index=False)

    print(f"Saved: {output_name} | shape: {cleaned_df.shape}")

Saved: 01_premier_league_filtered.csv | shape: (1495, 7)
Saved: 02_la_liga_filtered.csv | shape: (1426, 7)
Saved: 03_bundesliga_filtered.csv | shape: (1405, 7)
Saved: 04_ligue_1_filtered.csv | shape: (1320, 7)
Saved: 05_serie_a_filtered.csv | shape: (2339, 7)


# 2. Performance dataset - Single dataset with all 5 leagues

## Loading the DataFrame

In [7]:
df_perf = pd.read_csv("../data/raw/performance/01_team_performance.csv")

/var/folders/7q/2wc5cs997tqgvx852w8yrf1r0000gn/T/ipykernel_68926/298146210.py:1: DtypeWarning: Columns (3,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_perf = pd.read_csv("../data/raw/performance/01_team_performance.csv")


# Filtering dataset for scope

In [8]:
# seasons as per scope
valid_seasons = [
    "13-14", "14-15", "15-16", "16-17", "17-18",
    "18-19", "19-20", "20-21", "21-22", "22-23"
]

# leagues as per scope
valid_leagues = [
    "Premier League",
    "LaLiga",
    "Serie A",
    "Ligue 1",
    "Bundesliga"
]

# filter seasons
df_perf = df_perf[df_perf["Season"].isin(valid_seasons)]

# filter leagues
df_perf = df_perf[df_perf["Div"].isin(valid_leagues)]

# keep only required columns
df_perf = df_perf[
    ["Season", "Div", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
].copy()

# create total goals column
df_perf["total_goals"] = df_perf["FTHG"] + df_perf["FTAG"]

# preview cleaned data
df_perf.head()

,Season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,total_goals
0,22-23,Premier League,05/08/2022,Crystal Palace,Arsenal,0.0,2.0,A,2.0
1,22-23,Premier League,06/08/2022,Fulham,Liverpool,2.0,2.0,D,4.0
2,22-23,Premier League,06/08/2022,Bournemouth,Aston Villa,2.0,0.0,H,2.0
3,22-23,Premier League,06/08/2022,Leeds,Wolves,2.0,1.0,H,3.0
4,22-23,Premier League,06/08/2022,Newcastle,Nott'm Forest,2.0,0.0,H,2.0


- FTHG - Fill Time Home Goals
- FTAG - Full Time Away Goals
- FTR - Full Time Result

In [9]:
df_perf["Div"].value_counts()

Div
Premier League    3800
Serie A           3800
LaLiga            3800
Ligue 1           3699
Bundesliga        3060
Name: count, dtype: int64

# 3. Transfer and Performance dataset - both combined

## Loading the DataFrame

In [10]:
df_clubs = pd.read_csv("../data/raw/transf_perf/clubs.csv")
df_comps = pd.read_csv("../data/raw/transf_perf/competitions.csv")
df_games = pd.read_csv("../data/raw/transf_perf/games.csv")
df_transfers = pd.read_csv("../data/raw/transf_perf/transfers.csv")

## Joining datasets and filtering for scope

In [11]:
def build_team_performance(df_games: pd.DataFrame) -> pd.DataFrame:
    """
    Build a team-season performance table from match-level game data.

    Output:
        one row per club_id x season x competition_id
        with points, wins, draws, losses, goals_for, goals_against,
        goal_difference, and matches_played
    """

    # Keep only columns needed for performance aggregation
    games = df_games[
        [
            "game_id",
            "competition_id",
            "season",
            "home_club_id",
            "away_club_id",
            "home_club_name",
            "away_club_name",
            "home_club_goals",
            "away_club_goals",
        ]
    ].copy()

    # Build home-team view 
    home = games.rename(
        columns={
            "home_club_id": "club_id",
            "home_club_name": "club_name",
            "home_club_goals": "goals_for",
            "away_club_goals": "goals_against",
        }
    )[
        [
            "game_id",
            "competition_id",
            "season",
            "club_id",
            "club_name",
            "goals_for",
            "goals_against",
        ]
    ].copy()

    # Assign match result for home team
    home["win"] = (home["goals_for"] > home["goals_against"]).astype(int)
    home["draw"] = (home["goals_for"] == home["goals_against"]).astype(int)
    home["loss"] = (home["goals_for"] < home["goals_against"]).astype(int)
    home["points"] = home["win"] * 3 + home["draw"]

    # Build away-team view 
    away = games.rename(
        columns={
            "away_club_id": "club_id",
            "away_club_name": "club_name",
            "away_club_goals": "goals_for",
            "home_club_goals": "goals_against",
        }
    )[
        [
            "game_id",
            "competition_id",
            "season",
            "club_id",
            "club_name",
            "goals_for",
            "goals_against",
        ]
    ].copy()

    # Assign match result for away team
    away["win"] = (away["goals_for"] > away["goals_against"]).astype(int)
    away["draw"] = (away["goals_for"] == away["goals_against"]).astype(int)
    away["loss"] = (away["goals_for"] < away["goals_against"]).astype(int)
    away["points"] = away["win"] * 3 + away["draw"]

    # Stack home and away rows into one long team-match table 
    team_matches = pd.concat([home, away], ignore_index=True)

    # Aggregate to team-season level 
    team_perf = (
        team_matches.groupby(
            ["season", "competition_id", "club_id", "club_name"], as_index=False
        )
        .agg(
            matches_played=("game_id", "count"),
            wins=("win", "sum"),
            draws=("draw", "sum"),
            losses=("loss", "sum"),
            goals_for=("goals_for", "sum"),
            goals_against=("goals_against", "sum"),
            points=("points", "sum"),
        )
    )

    # Create derived metrics
    team_perf["goal_difference"] = (
        team_perf["goals_for"] - team_perf["goals_against"]
    )

    # Add league position within each season x competition 
    # Standard ranking logic:
    # points desc, goal difference desc, goals_for desc
    team_perf = team_perf.sort_values(
        by=["season", "competition_id", "points", "goal_difference", "goals_for"],
        ascending=[True, True, False, False, False],
    ).copy()

    team_perf["position"] = (
        team_perf.groupby(["season", "competition_id"])
        .cumcount()
        .add(1)
    )

    return team_perf.reset_index(drop=True)

In [12]:
df_team_perf = build_team_performance(df_games)
df_team_perf.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position
0,2012,BE1,58,Royal Sporting Club Anderlecht,30,20,7,3,69,27,67,42,1
1,2012,BE1,3508,Sportvereniging Zulte Waregem,30,19,6,5,49,29,63,20,2
2,2012,BE1,1184,Koninklijke Racing Club Genk,30,15,10,5,63,40,55,23,3
3,2012,BE1,2282,Club Brugge Koninklijke Voetbalvereniging,30,15,9,6,66,43,54,23,4
4,2012,BE1,498,KSC Lokeren (- 2020),30,14,9,7,53,38,51,15,5


In [29]:
df_team_perf = build_team_performance(df_games)
print(df_team_perf.shape)
print(sorted(df_team_perf["competition_id"].astype(str).unique()))

(20463, 13)
['BE1', 'BESC', 'CDR', 'CGB', 'CIT', 'CL', 'CLQ', 'DFB', 'DFL', 'DK1', 'DKP', 'ECLQ', 'EL', 'ELQ', 'ES1', 'FAC', 'FR1', 'FRCH', 'GB1', 'GBCS', 'GR1', 'GRP', 'IT1', 'KLUB', 'L1', 'NL1', 'NLP', 'NLSC', 'PO1', 'POCP', 'POSU', 'RU1', 'RUP', 'RUSS', 'SC1', 'SCI', 'SFA', 'SUC', 'TR1', 'UCOL', 'UKR1', 'UKRP', 'UKRS', 'USC']


## Only keep major 5 leagues

In [30]:
TOP5_IDS = ["GB1", "ES1", "IT1", "FR1", "L1"]

df_team_perf = df_team_perf[
    df_team_perf["competition_id"].astype(str).isin(TOP5_IDS)
].copy()

print(df_team_perf.shape)
print(df_team_perf["competition_id"].value_counts())

(1366, 13)
competition_id
ES1    280
GB1    280
IT1    280
FR1    274
L1     252
Name: count, dtype: int64


In [31]:
def attach_competition_meta(df_team_perf: pd.DataFrame, df_comps: pd.DataFrame) -> pd.DataFrame:
    """Attach readable competition metadata."""
    comps = df_comps[["competition_id", "competition_code", "name", "country_name"]].copy()
    comps["competition_id"] = comps["competition_id"].astype(str)

    perf = df_team_perf.copy()
    perf["competition_id"] = perf["competition_id"].astype(str)

    return perf.merge(comps, on="competition_id", how="left")


df_team_perf = attach_competition_meta(df_team_perf, df_comps)
print(df_team_perf.shape)
print(df_team_perf[["competition_id", "competition_code"]].drop_duplicates().sort_values("competition_id"))

(1366, 16)
   competition_id competition_code
0             ES1           laliga
20            FR1          ligue-1
40            GB1   premier-league
60            IT1          serie-a
80             L1       bundesliga


## Creating a Transfer spending dataset

In [52]:
def build_transfer_spending(df_transfers: pd.DataFrame) -> pd.DataFrame:
    """Build club-season transfer spending with season as start year."""

    transfers = df_transfers[
        ["to_club_id", "transfer_season", "transfer_fee"]
    ].copy()

    transfers = transfers.rename(columns={"to_club_id": "club_id"})

    # extract first year of season
    season_start = transfers["transfer_season"].astype(str).str.split("/").str[0]

    transfers["season"] = season_start.astype(int)
    transfers.loc[transfers["season"] < 100, "season"] += 2000

    # keep only seasons relevant for the project
    transfers = transfers[
        (transfers["season"] >= 2013) &
        (transfers["season"] <= 2022)
    ]

    # clean transfer fees
    transfers["transfer_fee"] = pd.to_numeric(
        transfers["transfer_fee"], errors="coerce"
    )

    transfers = transfers.dropna(subset=["transfer_fee"])

    transfer_spending = (
        transfers.groupby(["club_id", "season"], as_index=False)
        .agg(transfer_spending=("transfer_fee", "sum"))
    )

    return transfer_spending

In [53]:
df_transfer_spending = build_transfer_spending(df_transfers)

df_transfer_spending["season"].min(), df_transfer_spending["season"].max()

(2013, 2022)

### Merging transfer spend with team performance

In [54]:
df_final = df_team_perf.merge(
    df_transfer_spending,
    on=["club_id", "season"],
    how="left"
)

df_final["transfer_spending"] = df_final["transfer_spending"].fillna(0)

In [56]:
df_final["transfer_spending"].describe()

count    1.366000e+03
mean     2.394665e+07
std      4.513839e+07
min      0.000000e+00
25%      0.000000e+00
50%      4.250000e+06
75%      2.730000e+07
max      5.743500e+08
Name: transfer_spending, dtype: float64

In [58]:
df_final.shape

(1366, 17)

In [61]:
df_final.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name,transfer_spending
0,2012,ES1,131,Futbol Club Barcelona,38,32,4,2,115,40,100,75,1,laliga,laliga,Spain,0.0
1,2012,ES1,418,Real Madrid Club de Fútbol,38,26,7,5,103,42,85,61,2,laliga,laliga,Spain,0.0
2,2012,ES1,13,Club Atlético de Madrid S.A.D.,38,23,7,8,65,31,76,34,3,laliga,laliga,Spain,0.0
3,2012,ES1,681,Real Sociedad de Fútbol S.A.D.,38,18,12,8,70,49,66,21,4,laliga,laliga,Spain,0.0
4,2012,ES1,1049,Valencia Club de Fútbol S. A. D.,38,19,8,11,67,54,65,13,5,laliga,laliga,Spain,0.0


In [63]:
df_final["transfer_spending"].min(), df_final["transfer_spending"].max()

(0.0, 574350000.0)

## Saving datasets

In [68]:
df_final.to_csv(
    "../data/intermediate/transf_perf.csv",
    index=False
)

df_team_perf.to_csv(
    "../data/intermediate/team_performance.csv",
    index=False
)

df_transfer_spending.to_csv(
    "../data/intermediate/transfer_spending.csv",
    index=False
)

In [73]:
df_final.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name,transfer_spending
0,2012,ES1,131,Futbol Club Barcelona,38,32,4,2,115,40,100,75,1,laliga,laliga,Spain,0.0
1,2012,ES1,418,Real Madrid Club de Fútbol,38,26,7,5,103,42,85,61,2,laliga,laliga,Spain,0.0
2,2012,ES1,13,Club Atlético de Madrid S.A.D.,38,23,7,8,65,31,76,34,3,laliga,laliga,Spain,0.0
3,2012,ES1,681,Real Sociedad de Fútbol S.A.D.,38,18,12,8,70,49,66,21,4,laliga,laliga,Spain,0.0
4,2012,ES1,1049,Valencia Club de Fútbol S. A. D.,38,19,8,11,67,54,65,13,5,laliga,laliga,Spain,0.0


In [77]:
df_final[df_final["transfer_spending"] > 0].head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name,transfer_spending
98,2013,ES1,13,Club Atlético de Madrid S.A.D.,38,28,6,4,77,26,90,51,1,laliga,laliga,Spain,24995000.0
100,2013,ES1,418,Real Madrid Club de Fútbol,38,27,6,5,104,38,87,66,3,laliga,laliga,Spain,42500000.0
102,2013,ES1,368,Sevilla Fútbol Club S.A.D.,38,18,9,11,69,52,63,17,5,laliga,laliga,Spain,14900000.0
103,2013,ES1,1050,Villarreal Club de Fútbol S.A.D.,38,17,8,13,60,44,59,16,6,laliga,laliga,Spain,3300000.0
105,2013,ES1,1049,Valencia Club de Fútbol S. A. D.,38,13,10,15,51,53,49,-2,8,laliga,laliga,Spain,2050000.0


In [65]:
df_team_perf.head()

,season,competition_id,club_id,club_name,matches_played,wins,draws,losses,goals_for,goals_against,points,goal_difference,position,competition_code,name,country_name
0,2012,ES1,131,Futbol Club Barcelona,38,32,4,2,115,40,100,75,1,laliga,laliga,Spain
1,2012,ES1,418,Real Madrid Club de Fútbol,38,26,7,5,103,42,85,61,2,laliga,laliga,Spain
2,2012,ES1,13,Club Atlético de Madrid S.A.D.,38,23,7,8,65,31,76,34,3,laliga,laliga,Spain
3,2012,ES1,681,Real Sociedad de Fútbol S.A.D.,38,18,12,8,70,49,66,21,4,laliga,laliga,Spain
4,2012,ES1,1049,Valencia Club de Fútbol S. A. D.,38,19,8,11,67,54,65,13,5,laliga,laliga,Spain


In [66]:
df_transfer_spending.head()

,club_id,season,transfer_spending
0,1,2015,0.0
1,1,2017,0.0
2,1,2022,0.0
3,2,2013,630000.0
4,2,2014,1000000.0
